# MeanReversion Per-Asset PBO-Safe Optimization

Optimize MeanReversion v2 z-score hyperparameters per asset with PBO (CSCV) validation.

**Assets**: BTCUSDT 1h, BTCUSDT 4h, ETHUSDT 4h, XRPUSDT 1h, BNBUSDT 30m, DOGEUSDT 4h  
**Data window**: 2024-06-01 → 2026-06-01 (2 years)  
**PBO threshold**: < 0.50 = GO  
**Scoring**: `compute_signal_weighted_returns()` (continuous edge → proportional position)  
**Transaction cost**: 10 bps (1h/4h), 15 bps (30m BNBUSDT)

In [ ]:
# ── Cell 1: Imports and Setup ──────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import math
import time
import warnings
from datetime import datetime, timezone
from scipy import stats
from tqdm import tqdm

from binance.um_futures import UMFutures

# Indicator batch functions
from libs.features.indicators.momentum.rsi import _compute_rsi_batch
from libs.features.indicators.volatility.bollinger import _compute_bb_batch
from libs.features.indicators.volatility.atr import _compute_atr_batch
from libs.features.indicators.trend.kama import _compute_kama_batch
from libs.features.indicators.momentum.adx import _compute_adx_batch

# Model & scoring
from libs.models.mean_reversion.model import MeanReversionModel
from libs.optim_utils.scoring import (
    compute_signal_weighted_returns,
    compute_sharpe,
    compute_max_drawdown,
    BARS_PER_YEAR,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
client = UMFutures()

print('Setup complete')

In [ ]:
# ── Cell 2: Data Fetching (all 6 asset/TF pairs) ──────────────────

_ALL_COLS = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore'
]

def fetch_ohlcv(symbol: str, interval: str, start_date: str, end_date: str) -> pd.DataFrame:
    start_ms = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ms = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    all_rows = []
    cursor = start_ms
    while cursor < end_ms:
        raw = client.klines(symbol, interval, startTime=cursor, endTime=end_ms, limit=1500)
        if not raw:
            break
        all_rows.extend(raw)
        cursor = int(raw[-1][6]) + 1
        if len(raw) < 1500:
            break
        time.sleep(0.15)
    df = pd.DataFrame(all_rows, columns=_ALL_COLS)
    for c in ['open', 'high', 'low', 'close', 'volume']:
        df[c] = df[c].astype(float)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df = df.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)
    print(f'{symbol} {interval}: {len(df)} bars [{df.timestamp.iloc[0]} → {df.timestamp.iloc[-1]}]')
    return df

# Asset/TF pairs from production config
ASSET_TF_PAIRS = [
    ('BTCUSDT', '1h'),
    ('BTCUSDT', '4h'),
    ('ETHUSDT', '4h'),
    ('XRPUSDT', '1h'),
    ('BNBUSDT', '30m'),
    ('DOGEUSDT', '4h'),
]

START, END = '2024-06-01', '2026-06-01'

# Cost and annualization by timeframe
COST_BPS = {'1h': 10.0, '4h': 10.0, '30m': 15.0}
BARS_PER_YEAR_MAP = {'1h': 8760, '4h': 2190, '30m': 17520}

data = {}
for sym, tf in ASSET_TF_PAIRS:
    key = f'{sym}_{tf}'
    data[key] = fetch_ohlcv(sym, tf, START, END)

print(f'\nFetched {len(data)} asset/TF pairs, total {sum(len(v) for v in data.values())} bars')

In [ ]:
# ── Cell 3: Feature Computation ───────────────────────────────────

def build_mr_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build feature DataFrame for MeanReversionModel.batch_evaluate().
    Computes: RSI, BollingerBands, KAMA_fast, ATR, ADX.
    """
    fdf = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']].copy()
    fdf = fdf.set_index('timestamp').sort_index()

    h = fdf['high'].values
    l = fdf['low'].values
    c = fdf['close'].values

    # RSI (14)
    fdf['RSI'] = _compute_rsi_batch(c, 14)

    # Bollinger Bands (20, 2.0)
    bb_mid, bb_upper, bb_lower = _compute_bb_batch(c, 20, 2.0)
    fdf['BollingerBands_upper'] = bb_upper
    fdf['BollingerBands_lower'] = bb_lower

    # KAMA fast (5, 2, 10) — matches production features.yaml
    fdf['KAMA_fast'] = _compute_kama_batch(c, 5, 2, 10)

    # ATR (14)
    fdf['ATR'] = _compute_atr_batch(h, l, c, 14)

    # ADX (14) — returns (adx, plus_di, minus_di)
    adx_arr, _, _ = _compute_adx_batch(h, l, c, 14)
    fdf['ADX_adx'] = adx_arr

    return fdf


feature_dfs = {}
for key, df in data.items():
    fdf = build_mr_features(df)
    feature_dfs[key] = fdf
    nans = fdf[['RSI', 'BollingerBands_upper', 'KAMA_fast', 'ATR', 'ADX_adx']].isna().sum()
    print(f'{key}: {len(fdf)} bars, NaN counts: {dict(nans)}')

print('\nFeature computation complete')

In [ ]:
# ── Cell 4: CSCV PBO Function ────────────────────────────────────

def cscv_pbo(returns_matrix: np.ndarray, S: int = 16, ann_factor: int = 2190) -> tuple:
    """
    Combinatorially Symmetric Cross-Validation for PBO.

    Args:
        returns_matrix: T x N matrix where each column is the return series
                       from one strategy/parameter configuration.
        S: Number of equal-sized subsamples to split data into (must be even).
        ann_factor: Bars per year for Sharpe annualization.

    Returns:
        pbo: Probability of backtest overfitting (0 to 1).
        logits: Array of logit values from each CSCV split.
    """
    assert S % 2 == 0, 'S must be even'
    T, N = returns_matrix.shape
    assert N >= 2, 'Need at least 2 configurations'

    block_size = T // S
    assert block_size > 0, f'Not enough rows ({T}) for S={S} sub-samples'
    trimmed = returns_matrix[: block_size * S]
    blocks = trimmed.reshape(S, block_size, N)

    half = S // 2
    combos = list(itertools.combinations(range(S), half))

    MAX_COMBOS = 15000
    if len(combos) > MAX_COMBOS:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(combos), MAX_COMBOS, replace=False)
        combos = [combos[i] for i in sorted(idx)]

    logits = []
    for train_idx in combos:
        test_idx = tuple(i for i in range(S) if i not in train_idx)

        train_returns = np.concatenate([blocks[i] for i in train_idx], axis=0)
        test_returns = np.concatenate([blocks[i] for i in test_idx], axis=0)

        train_sharpe = np.zeros(N)
        test_sharpe = np.zeros(N)
        for j in range(N):
            tr = train_returns[:, j]
            te = test_returns[:, j]
            std_tr = np.std(tr)
            std_te = np.std(te)
            train_sharpe[j] = (np.mean(tr) / std_tr * math.sqrt(ann_factor)) if std_tr > 1e-12 else 0.0
            test_sharpe[j] = (np.mean(te) / std_te * math.sqrt(ann_factor)) if std_te > 1e-12 else 0.0

        n_star = np.argmax(train_sharpe)
        oos_perf = test_sharpe[n_star]
        rank = np.sum(test_sharpe <= oos_perf)
        w = rank / N
        w = np.clip(w, 1e-6, 1.0 - 1e-6)
        logit = np.log(w / (1.0 - w))
        logits.append(logit)

    logits = np.array(logits)
    pbo = float(np.mean(logits <= 0))
    return pbo, logits


S = 16
PBO_THRESHOLD = 0.50
print(f'CSCV function defined.  S={S}, C({S},{S//2}) = {math.comb(S, S//2)} combinations per asset.')
print(f'PBO threshold: < {PBO_THRESHOLD} = GO')

In [ ]:
# ── Cell 5: Parameter Grid with Simplex Constraint ────────────────

GRID = {
    'rsi_scale':     [5, 10, 15, 20, 25, 30],
    'w_rsi':         [0.15, 0.3, 0.4, 0.5, 0.65],
    'w_bb':          [0.15, 0.3, 0.4, 0.5, 0.65],
    'adx_center':    [18, 25, 32, 40],
    'adx_steepness': [3, 5, 8, 12],
}

# Build all valid configs with simplex constraint: w_rsi + w_bb <= 1.0
all_mr_configs = []
for rsi_s in GRID['rsi_scale']:
    for w_rsi in GRID['w_rsi']:
        for w_bb in GRID['w_bb']:
            w_kama = round(1.0 - w_rsi - w_bb, 2)
            if w_kama < 0.0:
                continue  # Prune invalid combos
            for adx_c in GRID['adx_center']:
                for adx_s in GRID['adx_steepness']:
                    all_mr_configs.append({
                        'rsi_scale': float(rsi_s),
                        'w_rsi': float(w_rsi),
                        'w_bb': float(w_bb),
                        'w_kama': float(w_kama),
                        'adx_center': float(adx_c),
                        'adx_steepness': float(adx_s),
                    })

# Current production defaults (identical for all assets)
DEFAULT_MR = {
    'rsi_scale': 15.0, 'w_rsi': 0.4, 'w_bb': 0.4,
    'w_kama': 0.2, 'adx_center': 25.0, 'adx_steepness': 5.0,
}

print(f'Grid: {" x ".join(f"{k}({len(v)})" for k, v in GRID.items())}')
print(f'Valid configs after simplex pruning: {len(all_mr_configs)}')
print(f'Simplex check: w_kama range = [{min(c["w_kama"] for c in all_mr_configs)}, {max(c["w_kama"] for c in all_mr_configs)}]')

In [ ]:
# ── Cell 6: MR Returns Helper ────────────────────────────────────

def ann_sharpe(returns: np.ndarray, tf: str) -> float:
    """Annualized Sharpe using our BARS_PER_YEAR_MAP (handles 30m correctly)."""
    if len(returns) == 0 or np.std(returns) == 0:
        return 0.0
    ann = BARS_PER_YEAR_MAP.get(tf, 8760)
    return float((np.mean(returns) / np.std(returns)) * math.sqrt(ann))


def compute_mr_returns(
    feature_df: pd.DataFrame,
    params: dict,
    cost_bps: float = 10.0,
) -> np.ndarray:
    """
    Compute per-bar signal-weighted returns for a MeanReversion param config.

    Uses MeanReversionModel.batch_evaluate() → edge_scores,
    then compute_signal_weighted_returns() for proportional position sizing
    with transaction costs.

    Returns:
        np.ndarray of per-bar returns (length = len(feature_df) - 1)
    """
    model = MeanReversionModel(params)
    edge_scores = model.batch_evaluate(feature_df)
    close = feature_df['close'].values
    returns = compute_signal_weighted_returns(
        edge_scores.values, close, cost_bps=cost_bps,
    )
    return returns


def build_returns_matrix(
    feature_df: pd.DataFrame,
    configs: list[dict],
    cost_bps: float = 10.0,
    label: str = '',
) -> np.ndarray:
    """
    Build T x N returns matrix for CSCV PBO analysis.
    T = len(feature_df) - 1, N = len(configs).
    """
    T = len(feature_df) - 1  # returns are 1 shorter than prices
    N = len(configs)
    rm = np.zeros((T, N))

    for j, cfg in enumerate(tqdm(configs, desc=f'{label} grid')):
        try:
            ret = compute_mr_returns(feature_df, cfg, cost_bps=cost_bps)
            rm[:len(ret), j] = ret
        except Exception as e:
            pass  # Leave as zeros for failed configs

    return rm


print('MR returns helpers defined')

In [ ]:
# ── Cell 7: Per-Asset PBO Loop ───────────────────────────────────

pbo_results = {}
returns_matrices = {}

for sym, tf in ASSET_TF_PAIRS:
    key = f'{sym}_{tf}'
    fdf = feature_dfs[key]
    cost = COST_BPS[tf]
    ann = BARS_PER_YEAR_MAP[tf]

    print(f'\n{"="*60}')
    print(f'{key}  |  bars={len(fdf)}  |  cost={cost} bps  |  ann_factor={ann}')
    print(f'{"="*60}')

    # Adjust S if bars are too few for S=16
    s_val = S
    block_size = (len(fdf) - 1) // s_val
    if block_size < 200:
        s_val = 12
        block_size = (len(fdf) - 1) // s_val
        print(f'  Reduced S to {s_val} (block_size={block_size})')
    else:
        print(f'  S={s_val}, block_size={block_size}')

    # Build returns matrix
    rm = build_returns_matrix(fdf, all_mr_configs, cost_bps=cost, label=key)
    returns_matrices[key] = rm

    # Count active configs (produced non-zero returns)
    active_mask = np.any(rm != 0, axis=0)
    n_active = int(np.sum(active_mask))
    print(f'  Active configs: {n_active}/{len(all_mr_configs)}')

    if n_active < 10:
        print(f'  ⚠ Too few active configs for PBO — skipping')
        pbo_results[key] = {'pbo': float('nan'), 'verdict': 'SKIP', 'n_active': n_active}
        continue

    # Run CSCV on active configs only
    rm_active = rm[:, active_mask]
    pbo_val, logits = cscv_pbo(rm_active, S=s_val, ann_factor=ann)
    mean_logit = float(np.mean(logits))

    # Find best config on full sample
    full_sharpes = np.array([
        ann_sharpe(rm_active[:, j], tf) for j in range(rm_active.shape[1])
    ])
    best_idx = int(np.argmax(full_sharpes))
    best_sharpe = full_sharpes[best_idx]

    # Map back to config
    active_indices = np.where(active_mask)[0]
    best_cfg = all_mr_configs[active_indices[best_idx]]

    # Verdict
    if pbo_val < 0.40:
        verdict = 'GO'
    elif pbo_val < PBO_THRESHOLD:
        verdict = 'BORDERLINE-GO'
    else:
        verdict = 'NO-GO'

    pbo_results[key] = {
        'pbo': pbo_val,
        'mean_logit': mean_logit,
        'logits': logits,
        'best_sharpe': best_sharpe,
        'best_cfg': best_cfg,
        'verdict': verdict,
        'n_active': n_active,
        's_val': s_val,
    }

    print(f'  PBO = {pbo_val:.4f} | mean_logit = {mean_logit:.3f} | best_sharpe = {best_sharpe:.3f}')
    print(f'  Best config: {best_cfg}')
    print(f'  Verdict: {verdict}')

    # Early termination: if BTC 1h PBO > 0.60, warn
    if key == 'BTCUSDT_1h' and pbo_val > 0.60:
        print(f'\n  ⚠ EARLY TERMINATION WARNING: BTCUSDT 1h PBO={pbo_val:.4f} > 0.60')
        print(f'    MR may lack structural alpha. Consider reduced 2-param search.')

# Summary table
print(f'\n{"="*80}')
print('PBO RESULTS SUMMARY')
print(f'{"="*80}')
summary_rows = []
for key, r in pbo_results.items():
    summary_rows.append({
        'Asset_TF': key,
        'PBO': r['pbo'],
        'Mean_Logit': r.get('mean_logit', float('nan')),
        'Best_Sharpe': r.get('best_sharpe', float('nan')),
        'Active_Configs': r['n_active'],
        'Verdict': r['verdict'],
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
# ── Cell 8: Optuna Refinement (200 trials per GO asset) ──────────

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

optuna_results = {}

for sym, tf in ASSET_TF_PAIRS:
    key = f'{sym}_{tf}'
    r = pbo_results.get(key, {})
    if r.get('verdict') not in ('GO', 'BORDERLINE-GO'):
        print(f'{key}: {r.get("verdict", "SKIP")} — skipping Optuna')
        continue

    fdf = feature_dfs[key]
    cost = COST_BPS[tf]
    ann = BARS_PER_YEAR_MAP[tf]

    # Walk-forward split: 60% train, 20% val, 20% OOS
    n = len(fdf)
    purge = 24  # bars between segments
    t1 = int(n * 0.6)
    t2 = int(n * 0.8)
    train_fdf = fdf.iloc[:t1]
    val_fdf = fdf.iloc[t1 + purge:t2]

    def objective(trial, _fdf=val_fdf, _cost=cost, _tf=tf):
        rsi_scale = trial.suggest_float('rsi_scale', 5.0, 30.0, step=1.0)
        w_rsi = trial.suggest_float('w_rsi', 0.1, 0.8, step=0.05)
        w_bb = trial.suggest_float('w_bb', 0.1, min(0.8, 1.0 - w_rsi), step=0.05)
        w_kama = round(1.0 - w_rsi - w_bb, 2)
        if w_kama < 0.0:
            raise optuna.TrialPruned()
        adx_center = trial.suggest_float('adx_center', 15.0, 40.0, step=1.0)
        adx_steepness = trial.suggest_float('adx_steepness', 2.0, 15.0, step=1.0)

        params = {
            'rsi_scale': rsi_scale, 'w_rsi': w_rsi, 'w_bb': w_bb,
            'w_kama': w_kama, 'adx_center': adx_center, 'adx_steepness': adx_steepness,
        }

        ret = compute_mr_returns(_fdf, params, cost_bps=_cost)
        if len(ret) == 0 or np.std(ret) == 0:
            return -999.0

        sharpe = ann_sharpe(ret, _tf)
        mdd = compute_max_drawdown(ret)
        score = sharpe - 0.5 * abs(mdd)
        return score

    print(f'\n{key}: Running Optuna (200 trials)...')
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=200, show_progress_bar=True)

    best = study.best_params
    best['w_kama'] = round(1.0 - best['w_rsi'] - best['w_bb'], 2)

    optuna_results[key] = {
        'best_params': best,
        'best_score': study.best_value,
        'n_trials': len(study.trials),
    }

    print(f'  Best score: {study.best_value:.4f}')
    print(f'  Best params: {best}')

print(f'\nOptuna refinement complete for {len(optuna_results)} assets')

In [ ]:
# ── Cell 9: OOS Validation (Walk-Forward Degradation) ────────────

oos_results = {}

for key, opt_r in optuna_results.items():
    parts = key.split('_', 1)
    sym, tf = parts[0], parts[1]
    fdf = feature_dfs[key]
    cost = COST_BPS[tf]
    params = opt_r['best_params']

    n = len(fdf)
    purge = 24
    t1 = int(n * 0.6)
    t2 = int(n * 0.8)

    train_fdf = fdf.iloc[:t1]
    val_fdf = fdf.iloc[t1 + purge:t2]
    oos_fdf = fdf.iloc[t2 + purge:]

    # Compute returns on each segment
    train_ret = compute_mr_returns(train_fdf, params, cost_bps=cost)
    val_ret = compute_mr_returns(val_fdf, params, cost_bps=cost)
    oos_ret = compute_mr_returns(oos_fdf, params, cost_bps=cost)

    train_sharpe = ann_sharpe(train_ret, tf)
    val_sharpe = ann_sharpe(val_ret, tf)
    oos_sharpe = ann_sharpe(oos_ret, tf)

    train_mdd = compute_max_drawdown(train_ret)
    val_mdd = compute_max_drawdown(val_ret)
    oos_mdd = compute_max_drawdown(oos_ret)

    # IS = train+val combined for degradation calc
    is_sharpe = ann_sharpe(np.concatenate([train_ret, val_ret]), tf)

    # Degradation: how much does OOS Sharpe degrade vs IS?
    if abs(is_sharpe) > 1e-6:
        degradation = 1.0 - oos_sharpe / is_sharpe
    else:
        degradation = float('nan')

    # OOS verdict
    if oos_sharpe < -0.3:
        oos_verdict = 'NO-GO'
    elif degradation > 0.80:
        oos_verdict = 'NO-GO (>80% degradation)'
    elif degradation > 0.50:
        oos_verdict = 'WARNING (>50% degradation)'
    else:
        oos_verdict = 'PASS'

    oos_results[key] = {
        'params': params,
        'train_sharpe': train_sharpe,
        'val_sharpe': val_sharpe,
        'oos_sharpe': oos_sharpe,
        'is_sharpe': is_sharpe,
        'degradation': degradation,
        'train_mdd': train_mdd,
        'val_mdd': val_mdd,
        'oos_mdd': oos_mdd,
        'oos_verdict': oos_verdict,
    }

    print(f'\n{key}:')
    print(f'  Train Sharpe: {train_sharpe:.3f} | Val Sharpe: {val_sharpe:.3f} | OOS Sharpe: {oos_sharpe:.3f}')
    print(f'  IS Sharpe: {is_sharpe:.3f} | Degradation: {degradation:.1%}')
    print(f'  Train MDD: {train_mdd:.3%} | Val MDD: {val_mdd:.3%} | OOS MDD: {oos_mdd:.3%}')
    print(f'  OOS Verdict: {oos_verdict}')

In [ ]:
# ── Cell 10: Visualization ────────────────────────────────────────

go_assets = [k for k, r in pbo_results.items() if r.get('verdict') in ('GO', 'BORDERLINE-GO')]
n_plots = len(pbo_results)

# --- 10a: PBO bar chart ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PBO values
ax = axes[0]
keys_sorted = sorted(pbo_results.keys())
pbos = [pbo_results[k]['pbo'] for k in keys_sorted]
colors = ['green' if p < 0.40 else 'orange' if p < 0.50 else 'red' for p in pbos]
ax.barh(keys_sorted, pbos, color=colors)
ax.axvline(x=PBO_THRESHOLD, color='red', linestyle='--', label=f'Threshold={PBO_THRESHOLD}')
ax.axvline(x=0.40, color='orange', linestyle=':', label='GO boundary=0.40')
ax.set_xlabel('PBO')
ax.set_title('PBO per Asset/TF')
ax.legend()

# Mean logits
ax = axes[1]
mean_logits = [pbo_results[k].get('mean_logit', 0) for k in keys_sorted]
colors_l = ['green' if ml > 0 else 'red' for ml in mean_logits]
ax.barh(keys_sorted, mean_logits, color=colors_l)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Mean Logit')
ax.set_title('Mean CSCV Logit per Asset/TF')

plt.tight_layout()
plt.show()

# --- 10b: Logit distributions for GO assets ---
if go_assets:
    n_go = len(go_assets)
    fig, axes = plt.subplots(1, min(n_go, 3), figsize=(5 * min(n_go, 3), 4), squeeze=False)
    for i, key in enumerate(go_assets[:3]):
        ax = axes[0, i]
        logits = pbo_results[key]['logits']
        ax.hist(logits, bins=50, alpha=0.7, edgecolor='black')
        ax.axvline(x=0, color='red', linestyle='--')
        ax.set_title(f'{key}\nPBO={pbo_results[key]["pbo"]:.3f}')
        ax.set_xlabel('Logit')
    plt.suptitle('CSCV Logit Distributions (GO Assets)', fontsize=12)
    plt.tight_layout()
    plt.show()

# --- 10c: IS vs OOS Sharpe CDF for GO assets ---
if go_assets:
    fig, axes = plt.subplots(1, min(len(go_assets), 3), figsize=(5 * min(len(go_assets), 3), 4), squeeze=False)
    for i, key in enumerate(go_assets[:3]):
        ax = axes[0, i]
        rm = returns_matrices[key]
        active_mask = np.any(rm != 0, axis=0)
        rm_active = rm[:, active_mask]
        tf = key.split('_', 1)[1]

        # Half-sample split for IS/OOS CDF
        half_T = rm_active.shape[0] // 2
        is_sharpes = np.array([ann_sharpe(rm_active[:half_T, j], tf) for j in range(rm_active.shape[1])])
        oos_sharpes = np.array([ann_sharpe(rm_active[half_T:, j], tf) for j in range(rm_active.shape[1])])

        sorted_is = np.sort(is_sharpes)
        sorted_oos = np.sort(oos_sharpes)
        cdf = np.linspace(0, 1, len(sorted_is))
        ax.plot(sorted_is, cdf, label='IS', linewidth=2)
        ax.plot(sorted_oos, cdf, label='OOS', linewidth=2)
        ax.legend()
        ax.set_title(f'{key}')
        ax.set_xlabel('Sharpe')
        ax.set_ylabel('CDF')
    plt.suptitle('IS vs OOS Sharpe CDF', fontsize=12)
    plt.tight_layout()
    plt.show()

print('Visualization complete')

In [ ]:
# ── Cell 11: Final Summary Table ─────────────────────────────────

print('=' * 100)
print('FINAL SUMMARY: MeanReversion Per-Asset PBO Optimization')
print('=' * 100)

final_rows = []
for sym, tf in ASSET_TF_PAIRS:
    key = f'{sym}_{tf}'
    r = pbo_results.get(key, {})
    o = oos_results.get(key, {})
    opt = optuna_results.get(key, {})

    row = {
        'Asset_TF': key,
        'PBO': f"{r.get('pbo', float('nan')):.4f}",
        'Mean_Logit': f"{r.get('mean_logit', float('nan')):.3f}",
        'Grid_Verdict': r.get('verdict', 'N/A'),
        'Best_Grid_Sharpe': f"{r.get('best_sharpe', float('nan')):.3f}",
        'Optuna_Score': f"{opt.get('best_score', float('nan')):.4f}" if opt else 'N/A',
        'OOS_Sharpe': f"{o.get('oos_sharpe', float('nan')):.3f}" if o else 'N/A',
        'Degradation': f"{o.get('degradation', float('nan')):.1%}" if o else 'N/A',
        'OOS_Verdict': o.get('oos_verdict', 'N/A') if o else 'N/A',
    }
    final_rows.append(row)

final_df = pd.DataFrame(final_rows)
print(final_df.to_string(index=False))

# Print optimized params for GO assets
print('\n\n--- Optimized Params for GO/BORDERLINE-GO Assets ---')
for key, opt in optuna_results.items():
    oos_v = oos_results.get(key, {}).get('oos_verdict', 'N/A')
    print(f'\n{key} (OOS: {oos_v}):')
    for k, v in opt['best_params'].items():
        print(f'  {k}: {v}')

In [ ]:
# ── Cell 12: Config Output for models.yaml ───────────────────────

print('# Paste into configs/models.yaml under each asset\'s MeanReversion params')
print('# Only includes assets with PBO-GO + OOS PASS\n')

for sym, tf in ASSET_TF_PAIRS:
    key = f'{sym}_{tf}'
    r = pbo_results.get(key, {})
    o = oos_results.get(key, {})
    opt = optuna_results.get(key, {})

    if r.get('verdict') not in ('GO', 'BORDERLINE-GO'):
        print(f'# {key}: {r.get("verdict", "SKIP")} — keeping defaults')
        continue
    if o.get('oos_verdict') in ('NO-GO', 'NO-GO (>80% degradation)'):
        print(f'# {key}: OOS {o["oos_verdict"]} — keeping defaults')
        continue

    params = opt.get('best_params', {})
    if not params:
        continue

    print(f'# {key}: PBO={r["pbo"]:.4f} | OOS_Sharpe={o.get("oos_sharpe", 0):.3f} | {o.get("oos_verdict", "")}')
    print(f'#   {sym} / {tf} / MeanReversion / params:')
    print(f'          MeanReversion:')
    print(f'            enabled: true')
    print(f'            migration_mode: scoring')
    print(f'            params:')
    for k, v in params.items():
        print(f'              {k}: {v}')
    print()